# 21 · Final fine-tuning — Faster R-CNN Swin-T
Automatically loads notebook 11's best configuration and runs baseline/tuned seeds 17, 42, and 3407.

In [ ]:
DATASET_TRACK = "2class"
START_FINETUNING = False

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
IN_KAGGLE = not IN_COLAB and bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE")
    or Path("/kaggle/working").is_dir()
)
NOTEBOOK_PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

repository_default = (
    Path("/content/aerial-object-detection-benchmark")
    if IN_COLAB
    else Path("/kaggle/working/aerial-object-detection-benchmark")
    if IN_KAGGLE
    else Path.cwd()
)
repository_override = os.environ.get("BENCHMARK_REPO_ROOT")
repository_candidates = (
    [Path(repository_override).expanduser()]
    if repository_override
    else [Path.cwd(), *Path.cwd().parents, repository_default]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in repository_candidates
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "__init__.py").is_file()
    ),
    repository_default.resolve(),
)
git_probe = subprocess.run(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--is-inside-work-tree"],
    check=False, capture_output=True, text=True,
)
if git_probe.returncode != 0:
    if NOTEBOOK_PLATFORM == "local":
        raise RuntimeError(
            "Run this notebook from the repository or set BENCHMARK_REPO_ROOT."
        )
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)],
        check=True,
    )
else:
    current_commit = subprocess.check_output(
        ["git", "-C", str(REPO_PATH), "rev-parse", "HEAD"], text=True
    ).strip()
    print(f"Using selected repository commit {current_commit}.")
sys.path.insert(0, str(REPO_PATH))

from src.notebook_environment import setup_notebook_environment
notebook_environment = setup_notebook_environment(
    REPO_PATH,
    platform=NOTEBOOK_PLATFORM,
    use_google_drive=True,
    requirements_file="requirements-dataset-colab.txt",
    smoke_test=SMOKE_TEST,
)
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
LOCAL_CACHE_ROOT = notebook_environment.local_cache_root
print(notebook_environment.as_dict())


In [ ]:
MODEL_ID = "faster_rcnn_swin_t"
from src.models.swin_frcnn.trainer import SwinSharedTrainer
print(f"Shared training engine: {SwinSharedTrainer.__name__}")
if SMOKE_TEST:
    result = {"status": "guarded", "model_id": MODEL_ID}
else:
    from src.workflows.environment import ensure_model_environment
    from src.hpo.final_workflow import FinalExperimentWorkflow
    environment = (
        ensure_model_environment(MODEL_ID, REPO_PATH, DRIVE_ROOT)
        if START_FINETUNING
        else {"status": "SKIPPED_PREVIEW", "family": "openmmlab"}
    )
    result = FinalExperimentWorkflow(REPO_PATH, DRIVE_ROOT, MODEL_ID, DATASET_TRACK).run(start_expensive_stage=START_FINETUNING)
result